# 12 — Extracción exploratoria con OpenKOS sobre ES2015

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`08-jss-fechas-de-reunion.ipynb`](08-jss-fechas-de-reunion.ipynb) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) ·
[ADR 0006](../docs/decisions/0006-adr-c1-rag-separado-con-respuesta-de-openkos.md)

---

> **Qué es esto y qué no es.** Es una corrida **exploratoria** de la extracción automática de
> decisiones sobre **ES2015**, una serie de **desarrollo**. **No es un resultado del
> Experimento A.** Según el [ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md),
> no se reporta ninguna cifra de evaluación mientras la versión de OpenKOS no esté fijada
> ([ADR 0006](../docs/decisions/0006-adr-c1-rag-separado-con-respuesta-de-openkos.md)); la
> versión usada aquí, **0.5.5**, es provisional. Por eso este notebook solo **describe** las
> corridas: cuántas decisiones salen, cuánto se parecen entre sí y si repetir la misma semilla
> reproduce el resultado. No calcula precisión ni cobertura, ni cruza lo extraído con la
> referencia adjudicada.

---

# 1. Qué se corrió

OpenKOS procesa las cuatro reuniones de una serie (*a*, *b*, *c*, *d*) en orden, dentro de un
mismo espacio de trabajo, y al final entrega las decisiones que encontró, cada una con las
citas de la transcripción que la respaldan. Se hicieron dos corridas desde la línea de comandos
del proyecto (`afg experiment a`), guardadas en `reports/experiment-a/`:

- `es2015-explora-r1`: una corrida, con semilla 42.
- `es2015-explora-r3x`: tres corridas, con semillas 42, 43 y 44 (la semilla base más el número
  de corrida).

Las funciones que leen y resumen estas corridas están en `src/afg/experiments/extraction_runs.py`.

In [1]:
import pandas as pd

from afg.evaluation.reference import load_series_reference
from afg.experiments.extraction_runs import (
    decisions_matching,
    decisions_per_meeting,
    load_extraction_runs,
    pairwise_overlap,
    provenance_table,
    reference_counts_per_meeting,
    reproducibility_check,
)
from afg.shared.paths import AMI_DIR, EXPERIMENT_A_DIR, TRANSCRIPTS_DIR

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

RUN_DIRS = [EXPERIMENT_A_DIR / "es2015-explora-r1", EXPERIMENT_A_DIR / "es2015-explora-r3x"]
MEETINGS = ["ES2015a", "ES2015b", "ES2015c", "ES2015d"]

# Guard: the reference is read from the adjudicated CSV, the AMI summlink layer and the
# frozen transcripts.
for path in (*RUN_DIRS, AMI_DIR, TRANSCRIPTS_DIR):
    if not path.exists():
        raise SystemExit(f"No se encontro {path}.\nVer notebooks/README.md (Reproducibilidad).")

runs = load_extraction_runs(RUN_DIRS)
reference = load_series_reference("ES2015")
print(f"Corridas leídas: {len(runs)} -> {[r.label for r in runs]}")

Corridas leídas: 4 -> ['es2015-explora-r1 / corrida 0', 'es2015-explora-r3x / corrida 0', 'es2015-explora-r3x / corrida 1', 'es2015-explora-r3x / corrida 2']


---

# 2. Procedencia

Todo lo necesario para repetir la corrida queda en el manifiesto de cada carpeta: versión de
OpenKOS, modelos (con su *digest*, la huella que identifica la versión exacta de cada modelo
en Ollama), semilla, temperatura, commit del repositorio y las marcas de tiempo de inicio y
fin. La duración es la diferencia entre ambas marcas y cubre todas las corridas de la carpeta.

In [2]:
provenance_table(runs).T

,es2015-explora-r1,es2015-explora-r3x
versión de OpenKOS,0.5.5,0.5.5
modelo generativo,qwen3:8b,qwen3:8b
digest del modelo generativo,500a1f067a9f,500a1f067a9f
modelo de embeddings,bge-m3,bge-m3
digest del modelo de embeddings,790764642607,790764642607
temperatura,0.1,0.1
semilla base,42,42
corridas,1,3
commit,9d4687d528f0931eb37310807a9c839fbafb81a9,9d4687d528f0931eb37310807a9c839fbafb81a9
duración (min),25.7,75.0


**Resultado.** Ambas carpetas usan la misma versión de OpenKOS (0.5.5), los mismos modelos con
los mismos *digest*, la misma temperatura (0,1), la misma semilla base (42) y el mismo commit.
La corrida única tardó unos 26 minutos; las tres corridas, unos 75, es decir, unos 25 minutos
por corrida.

---

# 3. Decisiones extraídas por reunión, junto a la referencia

La referencia es el conjunto de decisiones adjudicadas de ES2015
(`ES2015.decisions.adjudicated.csv`): una fila cuenta como decisión si y solo si su estado final
es `decision`; en las decisiones compuestas cuentan los hijos, no el padre. La tabla pone lado
a lado lo extraído por cada corrida y esa referencia. **Es una comparación de cantidades, no de
contenido:** no dice si lo extraído coincide con lo adjudicado.

In [3]:
per_meeting = decisions_per_meeting(runs, MEETINGS)
table = per_meeting.pivot(
    index=["run_name", "run", "seed"], columns="meeting_id", values="n_decisions"
)
table["total"] = table.sum(axis=1)

ref_counts = reference_counts_per_meeting(reference, MEETINGS)
ref_row = pd.DataFrame(
    [[*ref_counts.tolist(), int(ref_counts.sum())]],
    columns=table.columns,
    index=pd.MultiIndex.from_tuples([("referencia adjudicada", "", "")], names=table.index.names),
)
pd.concat([table, ref_row])

meeting_id                      ES2015a  ES2015b  ES2015c  ES2015d  total
run_name              run seed                                           
es2015-explora-r1     0   42          4       22       25       20     71
es2015-explora-r3x    0   42          4       22       25       20     71
                      1   43          3       21       25       20     69
                      2   44          4       20       25       20     69
referencia adjudicada                 4        8        8        7     27

In [4]:
extracted_totals = table["total"]
print("Decisiones por corrida:", {f"{n}/{r}": int(t) for (n, r, _), t in extracted_totals.items()})
print("Decisiones de referencia:", int(ref_counts.sum()), "->", ref_counts.to_dict())
print("Filas excluidas de la referencia por estado final:", dict(reference.excluded))

Decisiones por corrida: {'es2015-explora-r1/0': 71, 'es2015-explora-r3x/0': 71, 'es2015-explora-r3x/1': 69, 'es2015-explora-r3x/2': 69}
Decisiones de referencia: 27 -> {'ES2015a': 4, 'ES2015b': 8, 'ES2015c': 8, 'ES2015d': 7}
Filas excluidas de la referencia por estado final: {'no_decision': 3, 'compuesta': 2, 'descartar': 1, 'sin_soporte': 1}


**Resultado.** Cada corrida extrae entre 69 y 71 decisiones, frente a las 27 de la referencia
adjudicada: unas dos veces y media más. La diferencia aparece en todas las reuniones salvo la
primera, donde ambas cifras son cercanas (3 o 4 extraídas frente a 4 de referencia). El
exceso es el hallazgo que importa para el Experimento A, porque apunta a la **precisión**; la
sección 6 muestra a qué se debe, al menos en parte. Con estas cantidades no se puede afirmar
cuántas de las extraídas son correctas.

---

# 4. Qué tanto se parecen las corridas entre sí

Dos extracciones se consideran la misma decisión si vienen de la misma reunión y su texto
(`content`) es idéntico. Para cada par de corridas se cuenta cuántas decisiones tienen en común
y el índice de Jaccard: las comunes divididas por el total de decisiones distintas entre ambas.
Vale 1 si las dos corridas entregan exactamente lo mismo y 0 si no comparten nada.

In [5]:
overlap = pairwise_overlap(runs)
overlap.round({"jaccard": 2})

,run_a,run_b,n_a,n_b,common,union,jaccard
0,es2015-explora-r1 / corrida 0,es2015-explora-r3x / corrida 0,71,71,71,71,1.00
1,es2015-explora-r1 / corrida 0,es2015-explora-r3x / corrida 1,71,69,59,81,0.73
2,es2015-explora-r1 / corrida 0,es2015-explora-r3x / corrida 2,71,69,63,77,0.82
3,es2015-explora-r3x / corrida 0,es2015-explora-r3x / corrida 1,71,69,59,81,0.73
4,es2015-explora-r3x / corrida 0,es2015-explora-r3x / corrida 2,71,69,63,77,0.82
5,es2015-explora-r3x / corrida 1,es2015-explora-r3x / corrida 2,69,69,57,81,0.70


In [6]:
# The three runs of es2015-explora-r3x have three different seeds (42, 43, 44).
in_r3x = overlap["run_a"].str.startswith("es2015-explora-r3x") & overlap["run_b"].str.startswith(
    "es2015-explora-r3x"
)
distinct = overlap[in_r3x]
print(
    f"Pares con semillas distintas: {len(distinct)}; en común: "
    f"{distinct['common'].tolist()}; Jaccard: {distinct['jaccard'].round(2).tolist()}"
)

Pares con semillas distintas: 3; en común: [59, 63, 57]; Jaccard: [0.73, 0.82, 0.7]


**Resultado.** Entre corridas con semillas distintas comparten 57 a 63 decisiones, con un
Jaccard de 0,70 a 0,82. Hay un núcleo estable, pero entre una cuarta parte y un tercio de lo
extraído cambia de una corrida a otra. Esa variación es una razón para que el Experimento A
use varias corridas por serie y reporte la dispersión entre ellas.

---

# 5. Reproducibilidad: misma semilla, mismo resultado

La corrida 0 de `es2015-explora-r1` y la corrida 0 de `es2015-explora-r3x` usaron la misma
semilla (42) y el mismo texto de entrada. Si el procedimiento es reproducible, deben entregar
exactamente las mismas decisiones.

In [7]:
r1_run0 = next(r for r in runs if r.run_name == "es2015-explora-r1")
r3x_run0 = next(r for r in runs if r.run_name == "es2015-explora-r3x" and r.run == 0)
check = reproducibility_check(r1_run0, r3x_run0)
print(f"Semilla: {check.seed}")
print(f"Decisiones en cada corrida: {check.n_decisions}")
print(f"¿Idénticas campo por campo? {check.identical}")

Semilla: 42
Decisiones en cada corrida: (71, 71)
¿Idénticas campo por campo? True


**Resultado.** Con la misma semilla, las dos corridas son idénticas, decisión por decisión y
campo por campo. La variación de la sección 4 se debe entonces a la semilla y no a
ruido no controlado del procedimiento: fijar la semilla hace repetible la extracción, que es lo
que el diseño necesita.

---

# 6. Observación cualitativa: no todo lo extraído es una decisión

Al leer las decisiones extraídas se ven elementos que no son decisiones en el sentido de la
guía de anotación: valoraciones, hechos o simples opciones sobre la mesa. Como esta es una serie
de desarrollo, es admisible mirar el contenido
([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)). Los ejemplos
siguientes, de la corrida 0, se buscan por su texto y se muestran tal como salieron.

In [8]:
EXAMPLES = r"was demonstrated|operating cost is|one of the options|good-looking"
decisions_matching(r1_run0, EXAMPLES)

,meeting_id,decision_object,content
0,ES2015d,leadership,good leadership was demonstrated
1,ES2015d,teamwork,good teamwork was demonstrated
2,ES2015d,budget,our operating cost is twelve twenty Euros
3,ES2015c,design theme for remote control,The fruits and vegetables theme is one of the options for the remote control
4,ES2015d,design of the remote,The remote should be good-looking


**Lectura.** «Good leadership was demonstrated» y «good teamwork was demonstrated» son
evaluaciones del desempeño del equipo, no acuerdos sobre el producto; «our operating cost is
twelve twenty Euros» es un dato; «the fruits and vegetables theme is one of the options» es una
opción que se menciona, no una elección. «The remote should be good-looking» es una
aspiración difusa y queda a criterio de la adjudicación si cuenta como decisión.

Estos casos son una **muestra ilustrativa**, no un recuento: no se midió qué fracción de las
decisiones extraídas es de este tipo. Plantean una pregunta de **precisión** (qué parte de lo
extraído es realmente una decisión), y esa pregunta es la que el Experimento A medirá contra la
referencia adjudicada, con la versión de OpenKOS ya fijada.

---

# 7. Conclusión

- Es una corrida exploratoria sobre una serie de desarrollo, con OpenKOS 0.5.5 provisional: no
  es un resultado del Experimento A.
- Cada corrida extrae 69 a 71 decisiones; la referencia adjudicada de ES2015 tiene 27.
- Entre semillas distintas, las corridas comparten 57 a 63 decisiones (Jaccard de 0,70 a 0,82).
- Con la misma semilla el resultado es idéntico: la extracción es reproducible.
- Parte del exceso respecto de la referencia son elementos que no son decisiones
  (evaluaciones, datos, opciones). Cuánto pesa es una pregunta de precisión que se deja al
  Experimento A.

Las cifras salen de las funciones de `src/afg/experiments/extraction_runs.py`.